# Radio Signal Parameter Estimation: Issues & Solutions Log

This notebook documents the process of training a physics-informed neural network to estimate the phase offset of a 16-QAM radio signal and reconstruct it. Below is a summary of the critical issues encountered during development and the solutions that led to a successful model (BER < 1%).

### 1. The "Symmetry Ambiguity" Problem

* **Issue:** The initial model converged to a loss of ~1.0 (random guessing) and a BER of ~0.85.
* **Root Cause:** 16-QAM is rotationally symmetric every $90^\circ$ ($\pi/2$). Without an external reference, the network cannot distinguish between true $\theta$ and $\theta+90^\circ$. It attempts to average these possibilities, leading to a destructive mean of 0.
* **Solution:** **Pilot-Aided Estimation**. We explicitly feed the first $N=4$ known symbols ("pilots") into the network. These act as a "phase anchor," breaking the symmetry and allowing the network to lock onto the correct quadrant.

### 2. The "Lazy Network" / Vanishing Gradient Problem

* **Issue:** Even with pilots, the network initially ignored them because the sparse pilot signal ($8$ floats) was drowned out by the noisy data signal ($1024$ floats).
* **Root Cause:** The network struggled to learn the specific complex-conjugate arithmetic required to extract phase from pilots from scratch (the optimization landscape was too flat).
* **Solution:** **Physics-Informed Residual Learning**. We calculate a "Classical Hint" (a rough Coarse Estimate using the pilots via Least Squares) and feed this vector $[\cos \hat{\theta}, \sin \hat{\theta}]$ into the network.
* *Result:* The NN no longer needs to learn the phase from zero; it only needs to learn the **Residual Correction** ($\Delta \theta$) to account for noise and non-linearities, which is a much easier task.



### 3. Input Normalization & SNR Curriculum

* **Issue:** Gradients were unstable, and the MLP struggled to converge on raw IQ data.
* **Solution:**
1. **Batch Normalization:** Added `BatchNorm1d` immediately after flattening inputs to center the data and keep variance unit-scale.
2. **SNR Randomization:** We trained on a dynamic SNR range ($15 - 30$ dB). This prevented the model from overfitting to clean signals and forced it to learn robust feature extraction for noisy environments.



### 4. Manifold-Aware Loss

* **Issue:** Using MSE (Mean Squared Error) on raw angles fails because $-\pi$ and $+\pi$ are far apart in Euclidean numbers but identical in physical phase (the "wrap-around" problem).
* **Solution:** We predict a vector $[\cos \theta, \sin \theta]$ and maximize the cosine similarity. The loss function is $L = 1 - \cos(\theta_{pred} - \theta_{true})$, which is differentiable and correctly respects the circular geometry of the phase manifold.

---

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import matplotlib.pyplot as plt
import random
import time
import torch.nn.functional as F

# -------------------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 50 
LR = 0.001
SEQ_LEN = 256
N_PILOTS = 4
DATA_PATH = "./data/radio_divina_commedia.pth"

class Colors:
    GREEN = '\033[92m'; RED = '\033[91m'; RESET = '\033[0m'; BOLD = '\033[1m'

print(f"Using device: {DEVICE}")

# -------------------------------------------------------------------------------------
# 2. Helpers
# -------------------------------------------------------------------------------------
def build_verified_constellation(path):
    ckpt = torch.load(path, map_location=DEVICE)
    iq_c = ckpt['iq_clean']
    z_c = torch.complex(iq_c[0,0,:], iq_c[0,1,:])
    z_np = z_c.cpu().numpy()
    const_np = np.unique(z_np)
    if len(const_np) < 16: 
        b = torch.tensor([-3.,-1.,1.,3.], device=DEVICE)
        g = torch.meshgrid(b, b, indexing='ij')
        const = (g[0] + 1j*g[1]).flatten()
        const = const / torch.sqrt(torch.mean(torch.abs(const)**2))
        return const.to(DEVICE)
    return torch.from_numpy(const_np).to(DEVICE)

CONST_TENSOR = build_verified_constellation(DATA_PATH)

def classical_pilot_estimate(noisy_pilots, clean_pilots):
    phasor = (noisy_pilots * torch.conj(clean_pilots)).sum(dim=1)
    norm = torch.abs(phasor) + 1e-8
    return torch.stack([phasor.real/norm, phasor.imag/norm], dim=1).float()

def demap_16qam(iq_tensor):
    dist = torch.abs(iq_tensor.unsqueeze(-1) - CONST_TENSOR.view(1, 1, -1))
    return torch.argmin(dist, dim=-1)

def calc_ber(pred_idx, true_idx):
    diff = pred_idx ^ true_idx
    b0 = (diff & 1); b1 = ((diff >> 1) & 1); b2 = ((diff >> 2) & 1); b3 = ((diff >> 3) & 1)
    return (b0 + b1 + b2 + b3).sum().float() / (pred_idx.numel() * 4)

def decode_text(indices, mask):
    try:
        ints = indices.cpu().numpy().astype(np.uint8)
        bits = np.unpackbits(ints[:, None], axis=1)[:, -4:].flatten()
        m = mask.cpu().numpy().flatten()[:len(bits)]
        clean = np.bitwise_xor(bits, m)
        return np.packbits(clean).tobytes().replace(b'\x00', b'').decode('utf-8', 'ignore')
    except: return "."

# -------------------------------------------------------------------------------------
# 3. Dataset
# -------------------------------------------------------------------------------------
def load_and_split_data(path, batch_size=BATCH_SIZE, seed=42):
    global SEQ_LEN
    print(f"Loading {path}...")
    ckpt = torch.load(path, map_location=DEVICE)
    x = ckpt['iq_noisy'].float()
    target_iq = ckpt['iq_clean'].float()
    
    y_phi = ckpt['phase_labels'].float().reshape(-1)
    y_cfo = ckpt['cfo_labels'].float().reshape(-1)
    y_snr = ckpt.get('snr_db', torch.zeros_like(y_phi)).float().reshape(-1)
    bits = ckpt['bits'] 
    masks = ckpt.get('scramble_mask', torch.zeros_like(bits))

    if x.shape[2] != SEQ_LEN: SEQ_LEN = x.shape[2]
    
    b_re = bits.reshape(bits.shape[0], -1, 4).long()
    z_idx = (b_re[:,:,0]*8 + b_re[:,:,1]*4 + b_re[:,:,2]*2 + b_re[:,:,3]*1)
    
    target_bits = bits.reshape(bits.shape[0], SEQ_LEN, 4).permute(0, 2, 1).float()
    
    ds = TensorDataset(x, target_iq, y_phi, y_cfo, y_snr, target_bits, z_idx, masks)
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    
    train_ds = Subset(ds, idx[:int(0.75*len(ds))])
    val_ds   = Subset(ds, idx[int(0.75*len(ds)):int(0.95*len(ds))])
    test_ds  = Subset(ds, idx[int(0.95*len(ds)):])
    
    return (DataLoader(train_ds, batch_size, shuffle=True), 
            DataLoader(val_ds, batch_size, False), 
            DataLoader(test_ds, batch_size, False))

train_loader, val_loader, test_loader = load_and_split_data(DATA_PATH)

# -------------------------------------------------------------------------------------
# 4. Architecture
# -------------------------------------------------------------------------------------

class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.dim_head = dim // heads
        self.scale = self.dim_head ** -0.5
        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Linear(dim, dim)
        self.film_gen = nn.Sequential(nn.Linear(4, dim * 2), nn.ReLU())

    def forward(self, x, physics_params):
        film = self.film_gen(physics_params)
        gamma, beta = film.chunk(2, dim=-1)
        x = x * (1.0 + gamma.unsqueeze(1)) + beta.unsqueeze(1)

        b, n, d = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: t.view(b, n, self.heads, self.dim_head).transpose(1, 2), qkv)

        q = F.elu(q) + 1.0
        k = F.elu(k) + 1.0

        kv = torch.matmul(k.transpose(-1, -2), v)
        z = 1.0 / (torch.matmul(q, k.transpose(-1, -2).sum(dim=-1).unsqueeze(-1)) + 1e-6)
        out = torch.matmul(q, kv) * z
        out = out.transpose(1, 2).reshape(b, n, d)
        
        return self.to_out(out)

class AttentiveResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU()
        )
        self.attn = LinearAttention(channels)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x, physics_params):
        res = x
        x = self.conv(x)
        x_attn = x.permute(0, 2, 1)
        x_attn = self.attn(self.norm(x_attn), physics_params)
        x = x + x_attn.permute(0, 2, 1)
        return x + res

class PeriodicityEstimator(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.conv_feat = nn.Sequential(
            nn.Conv1d(2, 16, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(16, 32, 5, stride=2, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) 
        )
        self.mlp_feat = nn.Sequential(nn.Linear(2*seq_len, 64), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(106, 128), nn.ReLU(),
            nn.Linear(128, 4)
        )
        
    def forward(self, x, pilots, hint):
        c = self.conv_feat(x).view(x.size(0), -1)   
        x_flat = x.view(x.size(0), -1)              
        m = self.mlp_feat(x_flat)                   
        p_flat = pilots.view(x.size(0), -1)         
        combined = torch.cat([c, m, p_flat, hint], dim=1)
        return self.head(combined)

# NEW: Transformer Decoder
class TransformerDemapper(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.input_proj = nn.Conv1d(2, dim, 1) # Clean IQ -> Dim
        self.pos_emb = nn.Parameter(torch.randn(1, 256, dim))
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=4, dim_feedforward=128, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.head = nn.Linear(dim, 4) # 4 Bits

    def forward(self, clean_iq):
        # clean_iq: [B, 2, L]
        x = self.input_proj(clean_iq).permute(0, 2, 1) # [B, L, Dim]
        x = x + self.pos_emb
        x = self.transformer(x)
        logits = self.head(x) # [B, L, 4]
        return logits.permute(0, 2, 1) # [B, 4, L]

class GuidedLCAR(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, n_pilots=N_PILOTS):
        super().__init__()
        
        self.estimator = PeriodicityEstimator(seq_len)
        
        self.refiner_in = nn.Conv1d(2, 64, 7, padding=3)
        self.attn1 = LinearAttention(64)
        self.attn2 = LinearAttention(64)
        self.refiner_out = nn.Conv1d(64, 2, 1) 
        
        # New Stronger Demapper
        self.demapper = TransformerDemapper(64)

    def forward(self, x_noisy, pilots, hint):
        params = self.estimator(x_noisy, pilots, hint)
        phi = torch.atan2(params[:, 1], params[:, 0])
        cfo = params[:, 2] / 10.0
        snr_est = params[:, 3]
        
        B, _, L = x_noisy.shape
        t = torch.arange(L, device=x_noisy.device).float().unsqueeze(0)
        phase_ramp = phi.unsqueeze(1) + (cfo.unsqueeze(1) * t)
        cos_t = torch.cos(phase_ramp).unsqueeze(1)
        sin_t = torch.sin(phase_ramp).unsqueeze(1)
        
        r_I = x_noisy[:,0,:] * cos_t.squeeze(1) + x_noisy[:,1,:] * sin_t.squeeze(1)
        r_Q = -x_noisy[:,0,:] * sin_t.squeeze(1) + x_noisy[:,1,:] * cos_t.squeeze(1)
        x_derot = torch.stack([r_I, r_Q], dim=1)
        
        feat = torch.relu(self.refiner_in(x_derot))
        feat = feat.permute(0, 2, 1)
        feat = self.attn1(feat, params) + feat
        feat = self.attn2(feat, params) + feat
        feat = feat.permute(0, 2, 1) 
        
        rec_iq = self.refiner_out(feat) 
        
        bit_logits = self.demapper(rec_iq)
        
        return bit_logits, rec_iq, phi, cfo, snr_est

# -------------------------------------------------------------------------------------
# 5. Training
# -------------------------------------------------------------------------------------
model = GuidedLCAR(SEQ_LEN, N_PILOTS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
loss_bce = nn.BCEWithLogitsLoss()
loss_mse = nn.MSELoss()

print(f"--- Starting Guided Training ({EPOCHS} Epochs) ---")

for epoch in range(EPOCHS):
    model.train()
    metrics = {'loss':0, 'bce':0, 'iq':0, 'phi':0, 'cfo':0, 'ber':0}
    
    if epoch < 10:
        mode = "LOCK (Phys)"
        w_phys, w_iq, w_bit = 1.0, 0.0, 0.0
    elif epoch < 25:
        mode = "CLEAN (IQ)"
        w_phys, w_iq, w_bit = 0.5, 10.0, 0.0 
    else:
        mode = "DECODE (Bits)"
        w_phys, w_iq, w_bit = 0.1, 1.0, 5.0 # High weight on bits

    for x, target_iq, y_p, y_c, y_s, target_bits, z_idx, _ in train_loader:
        x, target_iq = x.to(DEVICE), target_iq.to(DEVICE)
        y_p, y_c, y_s = y_p.to(DEVICE), y_c.to(DEVICE), y_s.to(DEVICE)
        target_bits, z_idx = target_bits.to(DEVICE), z_idx.to(DEVICE)
        
        p_ref = CONST_TENSOR[z_idx[:, :N_PILOTS]]
        hint = classical_pilot_estimate(torch.complex(x[:,0], x[:,1])[:, :N_PILOTS], p_ref)
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        
        bit_logits, rec_iq, phi, cfo, snr_est = model(x, p_feat, hint)
        
        l_phys = (1.0 - torch.cos(phi - y_p).mean()) + loss_mse(cfo*10, y_c*10) + loss_mse(snr_est, y_s/20.0)
        l_iq = loss_mse(rec_iq, target_iq)
        l_bit = loss_bce(bit_logits, target_bits)
        
        loss = (w_phys * l_phys) + (w_iq * l_iq) + (w_bit * l_bit)
        
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        
        with torch.no_grad():
            metrics['loss'] += loss.item()
            metrics['bce'] += l_bit.item()
            metrics['iq'] += l_iq.item()
            metrics['phi'] += torch.abs(torch.atan2(torch.sin(phi-y_p), torch.cos(phi-y_p))).mean().item()
            metrics['cfo'] += torch.abs(cfo - y_c).mean().item()
            
            pred_bits = (bit_logits > 0).long()
            pred_idx = (pred_bits[:,0,:]*8 + pred_bits[:,1,:]*4 + pred_bits[:,2,:]*2 + pred_bits[:,3,:]*1)
            metrics['ber'] += calc_ber(pred_idx, z_idx).item()

    model.eval()
    val_ber = 0
    with torch.no_grad():
        for x, _, _, _, _, _, z_idx, _ in val_loader:
            x, z_idx = x.to(DEVICE), z_idx.to(DEVICE)
            p_ref = CONST_TENSOR[z_idx[:, :N_PILOTS]]
            hint = classical_pilot_estimate(torch.complex(x[:,0], x[:,1])[:, :N_PILOTS], p_ref)
            p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
            
            bit_logits, _, _, _, _ = model(x, p_feat, hint)
            pred_bits = (bit_logits > 0).long()
            pred_idx = (pred_bits[:,0,:]*8 + pred_bits[:,1,:]*4 + pred_bits[:,2,:]*2 + pred_bits[:,3,:]*1)
            val_ber += calc_ber(pred_idx, z_idx).item() * z_idx.numel()*4
            
    n = len(train_loader)
    print(f"Ep {epoch+1:02d} [{mode}] | Loss: {metrics['loss']/n:.3f} | MSE(IQ): {metrics['iq']/n:.4f} | "
          f"Phi: {metrics['phi']/n:.3f} | CFO: {metrics['cfo']/n:.5f} | Val BER: {val_ber/(len(val_loader.dataset)*SEQ_LEN*4):.5f}")

# -------------------------------------------------------------------------------------
# 6. Benchmark
# -------------------------------------------------------------------------------------
print(f"\n{Colors.BOLD}{'ID':<3} | {'Method':<10} | {'Phase Err':<10} | {'BER':<8} | {'Text Reconstruction'}{Colors.RESET}")
print("-" * 150)

model.eval()
res = {'c_ber':[], 'n_ber':[]}
cnt = 0

with torch.no_grad():
    for x, _, y_p, _, _, _, z_idx, m in test_loader:
        x, z_idx, m = x.to(DEVICE), z_idx.to(DEVICE), m.to(DEVICE)
        x_c = torch.complex(x[:,0], x[:,1])
        
        p_ref = CONST_TENSOR[z_idx[:, :N_PILOTS]]
        hint = classical_pilot_estimate(x_c[:, :N_PILOTS], p_ref)
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        
        # Classical
        theta = torch.angle((x_c[:, :N_PILOTS] * torch.conj(p_ref)).sum(dim=1))
        rx_cl = x_c * torch.exp(-1j * theta.unsqueeze(1))
        idx_cl = demap_16qam(rx_cl)
        
        # Neural
        bit_logits, _, phi_n, _, _ = model(x, p_feat, hint)
        pred_bits = (bit_logits > 0).long()
        idx_nn = (pred_bits[:,0,:]*8 + pred_bits[:,1,:]*4 + pred_bits[:,2,:]*2 + pred_bits[:,3,:]*1)
        
        for i in range(x.size(0)):
            res['c_ber'].append(calc_ber(idx_cl[i], z_idx[i]).item())
            res['n_ber'].append(calc_ber(idx_nn[i], z_idx[i]).item())
            
            if cnt < 5:
                pay = slice(N_PILOTS, None)
                truth = decode_text(z_idx[i, pay], m[i, N_PILOTS*4:])
                txt_c = decode_text(idx_cl[i, pay], m[i, N_PILOTS*4:])
                txt_n = decode_text(idx_nn[i, pay], m[i, N_PILOTS*4:])
                def hl(t, p): return f"{Colors.GREEN}{p}{Colors.RESET}" if t==p else f"{t[:15]}.. vs {Colors.RED}{p[:15]}..{Colors.RESET}"
                print(f"{cnt+1:<3} | Classical  | {abs(theta[i].item() - y_p[i].item()):.4f} rad | {res['c_ber'][-1]:.4f}   | {hl(truth, txt_c)}")
                print(f"{'':<3} | Neural Net | {abs(phi_n[i].item() - y_p[i].item()):.4f} rad | {res['n_ber'][-1]:.4f}   | {hl(truth, txt_n)}")
                print("-" * 150)
                cnt += 1

print(f"Classical Mean BER: {np.mean(res['c_ber']):.5f}")
print(f"Neural Net Mean BER: {np.mean(res['n_ber']):.5f}")

Using device: cpu
Loading ./data/radio_divina_commedia.pth...
--- Starting Guided Training (50 Epochs) ---
Ep 01 [LOCK (Phys)] | Loss: 0.773 | MSE(IQ): 0.5095 | Phi: 0.936 | CFO: 0.00597 | Val BER: 0.50063
Ep 02 [LOCK (Phys)] | Loss: 0.416 | MSE(IQ): 0.5038 | Phi: 0.613 | CFO: 0.00543 | Val BER: 0.50070
Ep 03 [LOCK (Phys)] | Loss: 0.362 | MSE(IQ): 0.5023 | Phi: 0.551 | CFO: 0.00526 | Val BER: 0.50057
Ep 04 [LOCK (Phys)] | Loss: 0.303 | MSE(IQ): 0.5016 | Phi: 0.476 | CFO: 0.00521 | Val BER: 0.50068
Ep 05 [LOCK (Phys)] | Loss: 0.268 | MSE(IQ): 0.5007 | Phi: 0.438 | CFO: 0.00516 | Val BER: 0.50070
Ep 06 [LOCK (Phys)] | Loss: 0.223 | MSE(IQ): 0.4998 | Phi: 0.375 | CFO: 0.00514 | Val BER: 0.50068
Ep 07 [LOCK (Phys)] | Loss: 0.194 | MSE(IQ): 0.4997 | Phi: 0.338 | CFO: 0.00512 | Val BER: 0.50072
Ep 08 [LOCK (Phys)] | Loss: 0.171 | MSE(IQ): 0.4991 | Phi: 0.308 | CFO: 0.00511 | Val BER: 0.50068
Ep 09 [LOCK (Phys)] | Loss: 0.145 | MSE(IQ): 0.4988 | Phi: 0.277 | CFO: 0.00513 | Val BER: 0.50071
Ep